In [0]:
# Criando o schema 'gold' para as métricas de negócio
spark.sql("CREATE SCHEMA IF NOT EXISTS projeto_lakeflow.gold")

# Lendo os dados da camada Silver com Spark
TABELA_SILVER = "projeto_lakeflow.silver.claims_limpo"
df_silver = spark.table(TABELA_SILVER)

# Transformação Gold: Agregação contagem de claims por tipo de colisão (type)
df_gold = df_silver.groupBy("type").count().withColumnRenamed("count", "total_claims")

# Salvando a tabela agregada pronta para o consumo em relatórios
TABELA_GOLD = "projeto_lakeflow.gold.resumo_claims"
df_gold.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABELA_GOLD)

# Exibe o resultado do resumo analítico
display(df_gold)

In [0]:
from pyspark.sql import functions as F

df_silver = spark.table("projeto_lakeflow.silver.claims_limpo")

# Métrica 1: Custo médio do sinistro por gravidade (severity)
df_custo_gravidade = df_silver.groupBy("severity") \
    .agg(F.avg("total").alias("custo_medio"))

# Métrica 2: Volume de atividades suspeitas por tipo de colisão
df_suspeitos = df_silver.filter(F.col("suspicious_activity") == True) \
    .groupBy("collision_type") \
    .count().withColumnRenamed("count", "total_suspeitos")

# Salvando a Métrica 1 na camada Gold (custo por gravidade)
df_custo_gravidade.write.format("delta").mode("overwrite").saveAsTable("projeto_lakeflow.gold.custo_por_gravidade")

# Salvando a Métrica 2 na camada Gold (atividades suspeitas)
df_suspeitos.write.format("delta").mode("overwrite").saveAsTable("projeto_lakeflow.gold.atividades_suspeitas")

display(df_custo_gravidade)
display(df_suspeitos)
